# 03. Exploratory Data Analysis (EDA)

**Objective**: Conduct deep visual and statistical investigations into transaction behavior, answering the 5 business questions:
1. How many financial transactions are happening?
2. How many transactions are historically identified as fraudulent?
3. What patterns are associated with fraudulent transactions?
4. Which transaction categories, merchants, times, and locations show higher fraud rates?
5. What insights can financial analysts use to investigate suspicious patterns?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

# Set visualization aesthetic
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'Arial'

# Load processed Parquet dataset
df = pd.read_parquet('../data/processed/fraud_data.parquet')
print('Loaded dataset with records:', f'{len(df):,}')

In [ ]:
# 1. Fraud Distribution & Class Imbalance
counts = df['is_fraud'].value_counts()
fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(['Legitimate (0)', 'Fraudulent (1)'], counts.values, color=['#10B981', '#EF4444'])
ax.set_title('Overall Transaction Distribution (1,852,394 Records)', fontsize=13, fontweight='bold')
ax.set_ylabel('Number of Transactions')
for bar in bars:
    height = bar.get_height()
    ax.annotate(f'{height:,}', xy=(bar.get_x() + bar.get_width() / 2, height),
                xytext=(0, 3), textcoords='offset points', ha='center', va='bottom', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# 2. Category Analysis: Fraud Volume vs Fraud Rate
cat = df.groupby('category').agg(
    total=('is_fraud', 'count'),
    fraud=('is_fraud', 'sum')
).reset_index()
cat['fraud_rate'] = (cat['fraud'] / cat['total']) * 100
cat = cat.sort_values('fraud_rate', ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=cat, x='fraud_rate', y='category', palette='Reds_r', ax=ax)
ax.set_title('Historical Fraud Rate (%) by Transaction Category', fontsize=13, fontweight='bold')
ax.set_xlabel('Fraud Rate (%)')
plt.tight_layout()
plt.show()

In [ ]:
# 3. Diurnal (Hourly) Pattern of Fraud
hourly = df.groupby('hour').agg(total=('is_fraud', 'count'), fraud=('is_fraud', 'sum')).reset_index()
hourly['fraud_rate'] = (hourly['fraud'] / hourly['total']) * 100

fig, ax1 = plt.subplots(figsize=(10, 4))
ax1.plot(hourly['hour'], hourly['fraud'], color='#EF4444', marker='o', linewidth=2, label='Fraud Count')
ax1.set_xlabel('Hour of Day (00:00 - 23:00)')
ax1.set_ylabel('Fraud Transactions', color='#EF4444')
ax2 = ax1.twinx()
ax2.plot(hourly['hour'], hourly['fraud_rate'], color='#3B82F6', marker='s', linestyle='--', linewidth=2, label='Fraud Rate %')
ax2.set_ylabel('Fraud Rate (%)', color='#3B82F6')
plt.title('Hourly Fraud Volume and Rate Profile', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# 4. Amount Distribution: Legitimate vs Fraudulent
print('Average Legitimate Amount: $', f"{df[df['is_fraud']==0]['amt'].mean():.2f}")
print('Average Fraudulent Amount: $', f"{df[df['is_fraud']==1]['amt'].mean():.2f}")
print('Median Legitimate Amount:  $', f"{df[df['is_fraud']==0]['amt'].median():.2f}")
print('Median Fraudulent Amount:  $', f"{df[df['is_fraud']==1]['amt'].median():.2f}")

In [ ]:
# 5. Customer-to-Merchant Distance Analysis
dist = df.groupby('distance_bin', observed=False).agg(
    total=('is_fraud', 'count'),
    fraud=('is_fraud', 'sum')
).reset_index()
dist['fraud_rate'] = dist['fraud'] / dist['total'] * 100
print(dist)

### EDA Key Takeaways
- **Extreme Imbalance**: Only 0.521% of transactions are fraudulent.
- **Severe Amount Disparity**: Fraudulent transactions average $530.66 compared to $67.65 for legitimate transactions (7.84x disparity).
- **Night-Time Spike**: Fraud peaks intensely between 22:00 and 03:00, reaching a fraud rate of 2.60% at 22:00.
- **High-Risk Categories**: `shopping_net` (1.59%), `grocery_pos` (1.27%), and `misc_net` (0.97%) exhibit the highest vulnerability.